# MGC lead conversion baseline

This notebook builds a simple, explainable model that scores a lead **before the sales team calls it**. That prediction time determines which columns are allowed.

In [ ]:
# Run once if these packages are missing from your notebook environment.
# %pip install pandas scikit-learn

from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CSV_PATH = Path('leads.csv')
RANDOM_STATE = 42

## 1. Load and inspect

We parse the timestamp immediately and inspect the target balance, missing values, and duplicate identity hashes. Accuracy would be misleading if almost every record belongs to one class.

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=['created_at'])
print(f'Rows: {len(df):,}')
print('\nTarget counts:')
print(df['converted'].value_counts())
print(f"\nConversion rate: {df['converted'].mean():.2%}")
print(f"Duplicate hash groups: {df.loc[df.duplicated('crm_record_hash', keep=False), 'crm_record_hash'].nunique():,}")
print('\nMissing values:')
display(df.isna().sum().sort_values(ascending=False).head(8))

## 2. Cleaning and leakage decisions

- Normalize city spelling/case (`ISB` → `Islamabad`, `Rwp` → `Rawalpindi`, `khi` → `Karachi`).
- Sort by time and keep the first row per `crm_record_hash`, preventing the same person from appearing in both train and test.
- Drop `lead_id` and `crm_record_hash`: identifiers do not generalize to new leads.
- Drop `token_amount_received_pkr`: receiving a booking token happens at/after conversion and almost reveals the label directly.
- Drop `first_response_minutes`, `calls_made`, `total_call_seconds`, `whatsapp_replies`, and `site_visits`: these are post-contact outcomes unavailable when deciding whom to call first.
- Keep intake-time facts such as source, location, property preference, budget, overseas/referral/financing flags, and agent experience.
- Derive month, weekday, and hour from `created_at`; do not feed the raw timestamp or row order to the model.
- Impute numeric missing values with training medians and categorical missing values with the training mode inside the pipeline, avoiding test-data leakage.

In [ ]:
city_aliases = {'isb': 'islamabad', 'rwp': 'rawalpindi', 'khi': 'karachi'}
df['city'] = (df['city'].str.strip().str.lower().replace(city_aliases).str.title())
df = (df.sort_values('created_at')
        .drop_duplicates('crm_record_hash', keep='first')
        .copy())

df['created_month'] = df['created_at'].dt.month
df['created_dayofweek'] = df['created_at'].dt.dayofweek
df['created_hour'] = df['created_at'].dt.hour
print(f'Rows after deduplication: {len(df):,}')

## 3. Time-based train/test split

A random split lets future leads leak into training. Instead, the oldest 80% train the model and the newest 20% simulate deployment on later leads.

In [ ]:
categorical_features = ['source', 'city', 'area', 'property_type']
numeric_features = [
    'budget_pkr_lac', 'bedrooms', 'agent_experience_years',
    'is_overseas', 'referred_by_existing_client',
    'has_financing_approved', 'created_month',
    'created_dayofweek', 'created_hour'
]
features = categorical_features + numeric_features

split_at = int(len(df) * 0.80)
train = df.iloc[:split_at]
test = df.iloc[split_at:]
X_train, y_train = train[features], train['converted']
X_test, y_test = test[features], test['converted']

print(f'Train: {len(train):,} rows, {train.created_at.min()} to {train.created_at.max()}')
print(f'Test:  {len(test):,} rows, {test.created_at.min()} to {test.created_at.max()}')
print(f'Test conversion rate: {y_test.mean():.3f}')

## 4. Preprocess and train

Logistic regression is a fast, explainable baseline and provides conversion probabilities. `class_weight='balanced'` prevents the 93% non-converted majority from dominating training.

In [ ]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
preprocessor = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_features),
    ('numeric', numeric_pipeline, numeric_features)
])
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ))
])
model.fit(X_train, y_train)
conversion_scores = model.predict_proba(X_test)[:, 1]

## 5. Evaluate with Average Precision

Average Precision (area under the precision-recall curve) focuses on ranking the rare positive class. It is more useful here than accuracy: an always-negative model would be about 92% accurate while finding no conversions. The positive-rate baseline for AP is 0.079 on this holdout.

In [ ]:
average_precision = average_precision_score(y_test, conversion_scores)
print(f'Average Precision: {average_precision:.3f}')
print(f'No-skill baseline:  {y_test.mean():.3f}')

scored_leads = test[['lead_id', 'created_at', 'source', 'converted']].copy()
scored_leads['conversion_score'] = conversion_scores
display(scored_leads.sort_values('conversion_score', ascending=False).head(10))

### Result

On the newest 1,800 deduplicated leads, the baseline achieved **0.209 Average Precision**, compared with the **0.079 no-skill baseline**. This is a useful first ranking model, not a calibrated promise that a customer will convert.